In [ ]:
import pickle
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix


# ============================================================================
# 13. PAPER FIGURES AND TABLES
# ============================================================================
# Every table/figure here is regenerated directly from this project's saved
# pipeline outputs (data/processed/results/*.csv) -- never retyped from
# chat or memory. If a number in the paper draft ever disagrees with a
# number in these output files, the output file is correct and the paper
# text needs fixing, not the other way around. Re-run this notebook any
# time an upstream notebook (04/05/07/08/09/10/11/12) is re-run, so the
# paper's tables/figures always match the latest pipeline outputs.
#
# Inputs come only from files already produced by 01-12; this notebook adds
# no new modeling or filtering logic of its own beyond aggregation/plotting.
# Confusion matrices are computed directly from the raw prediction CSVs
# (not from any precomputed confusion-matrix CSV from 04) so there's one
# code path to trust instead of two.
#
# Requires matplotlib. If not already installed: pip install matplotlib
#
# Produces, into data/processed/results/paper_assets/:
#   table_corpus_stats.csv
#   table_main_results_raw.csv
#   table_toefl_shortcut_deltas.csv
#   table_confound_accuracy.csv
#   fig_self_disclosure_rate.png
#   fig_topic_exclusivity_rate.png
#   fig_confusion_toefl_masked_svm.png / fig_confusion_toefl_fulltext_svm.png
#   fig_confusion_toefl_masked_llm.png / fig_confusion_toefl_fulltext_llm.png
#   fig_confusion_toefl_masked_llm_reasoning.png / fig_confusion_toefl_fulltext_llm_reasoning.png

BASE_DIR = Path("..").resolve()
RESULTS_DIR = BASE_DIR / "data/processed/results"
ASSETS_DIR = RESULTS_DIR / "paper_assets"
ASSETS_DIR.mkdir(parents=True, exist_ok=True)

COMBINED_CSV_PATH = BASE_DIR / "data/processed/combined_nli_masked.csv"
LANGS = ["DEU", "FRA", "ITA", "JPN", "KOR", "SPA", "TUR", "ZHO"]

# ----------------------------------------------------------------------
# Table 1: corpus statistics -- essay counts by dataset x split x language
# ----------------------------------------------------------------------
print("=== Table 1: corpus statistics ===")
if COMBINED_CSV_PATH.exists():
    combined = pd.read_csv(COMBINED_CSV_PATH)
    combined["dataset"] = combined["dataset"].astype(str).str.strip().str.upper()
    combined["split"] = combined["split"].astype(str).str.strip().str.lower()
    combined["label"] = combined["label"].astype(str).str.strip().str.upper()

    corpus_stats = (
        combined.groupby(["dataset", "split", "label"]).size().rename("n").reset_index()
        .pivot_table(index=["dataset", "split"], columns="label", values="n", fill_value=0)
        .reindex(columns=LANGS)
    )
    corpus_stats.to_csv(ASSETS_DIR / "table_corpus_stats.csv")
    print(corpus_stats)
    print(f"Saved: {ASSETS_DIR / 'table_corpus_stats.csv'}")
else:
    print(f"Skipping -- {COMBINED_CSV_PATH.name} not found.")

# ----------------------------------------------------------------------
# Table 2: main RAW results across every system x condition x partition
# ----------------------------------------------------------------------
print("\n=== Table 2: main RAW results ===")
metrics_path = RESULTS_DIR / "full_2x2_comparison_metrics.csv"
if metrics_path.exists():
    # 10 writes the literal string "n/a" as the SVM sentinel for prompt_format.
    # pandas' default na_values list includes "n/a", so a plain read_csv silently
    # turns it into NaN -- which then makes pivot_table silently DROP the SVM row
    # (pivot_table drops NaN index levels by default). Restore the sentinel string
    # so SVM rows survive downstream grouping/pivoting.
    metrics = pd.read_csv(metrics_path, keep_default_na=True)
    metrics["prompt_format"] = metrics["prompt_format"].fillna("n/a")
    main_raw = metrics[metrics["subset"] == "raw"][
        ["system", "text_condition", "prompt_format", "partition", "n", "accuracy", "macro_f1"]
    ].sort_values(["system", "prompt_format", "text_condition", "partition"])
    main_raw.to_csv(ASSETS_DIR / "table_main_results_raw.csv", index=False)
    print(main_raw.to_string(index=False))
    print(f"Saved: {ASSETS_DIR / 'table_main_results_raw.csv'}")
else:
    print(f"Skipping -- {metrics_path.name} not found. Run 10 first.")

# ----------------------------------------------------------------------
# Table 3: TOEFL11-anchored shortcut deltas (fulltext acc - masked acc),
# shown across all three subset tiers (raw/clean/doubly_clean) so the
# "stable across every later filter" claim in the paper has a table behind
# it, not just an assertion.
# ----------------------------------------------------------------------
print("\n=== Table 3: TOEFL11 shortcut deltas (raw/clean/doubly_clean) ===")
deltas_path = RESULTS_DIR / "full_2x2_shortcut_deltas.csv"
if deltas_path.exists():
    deltas = pd.read_csv(deltas_path)
    deltas["prompt_format"] = deltas["prompt_format"].fillna("n/a")  # see note above
    toefl_deltas = deltas[deltas["partition"] == "toefl11_test"]
    toefl_deltas_wide = (
        toefl_deltas.pivot_table(
            index=["system", "prompt_format"], columns="subset", values="shortcut_delta_accuracy"
        )
        .reindex(columns=["raw", "clean", "doubly_clean"])
    )
    toefl_deltas_wide.to_csv(ASSETS_DIR / "table_toefl_shortcut_deltas.csv")
    print(toefl_deltas_wide)
    print(f"Saved: {ASSETS_DIR / 'table_toefl_shortcut_deltas.csv'}")
else:
    print(f"Skipping -- {deltas_path.name} not found. Run 10 first.")

# ----------------------------------------------------------------------
# Figure 1: self-disclosure rate by language x partition (bar chart)
# ----------------------------------------------------------------------
print("\n=== Figure 1: self-disclosure rate by language ===")
leak_summary_path = RESULTS_DIR / "leakage_summary_by_partition_label.csv"
if leak_summary_path.exists():
    leak_summary = pd.read_csv(leak_summary_path)
    plot_df = leak_summary[leak_summary["partition"].isin(["toefl11_test", "iclev3_external_test"])]
    pivot = plot_df.pivot(index="label", columns="partition", values="pct_flagged").reindex(LANGS)

    fig, ax = plt.subplots(figsize=(7, 4))
    pivot.plot(kind="bar", ax=ax)
    ax.set_ylabel("% essays self-disclosing own L1 country/language")
    ax.set_xlabel("Native language (L1)")
    ax.legend(title="Partition")
    fig.tight_layout()
    out_path = ASSETS_DIR / "fig_self_disclosure_rate.png"
    fig.savefig(out_path, dpi=200)
    plt.close(fig)
    print(f"Saved: {out_path}")
else:
    print(f"Skipping -- {leak_summary_path.name} not found. Run 11 first.")

# ----------------------------------------------------------------------
# Figure 2: topic-exclusivity rate by language (bar chart, ICLE only)
# ----------------------------------------------------------------------
print("\n=== Figure 2: topic-exclusivity rate by language ===")
topic_by_label_path = RESULTS_DIR / "icle_topic_exclusivity_by_label.csv"
if topic_by_label_path.exists():
    topic_by_label = pd.read_csv(topic_by_label_path).set_index("label").reindex(LANGS)

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(topic_by_label.index, topic_by_label["pct_exclusive"])
    ax.set_ylabel("% ICLE essays with a topic >=80% exclusive to own L1")
    ax.set_xlabel("Native language (L1)")
    fig.tight_layout()
    out_path = ASSETS_DIR / "fig_topic_exclusivity_rate.png"
    fig.savefig(out_path, dpi=200)
    plt.close(fig)
    print(f"Saved: {out_path}")
else:
    print(f"Skipping -- {topic_by_label_path.name} not found. Run 12 first.")

# ----------------------------------------------------------------------
# Table 4: accuracy by confound status (self-disclosure, topic-exclusivity),
# both systems where available. Each source file already carries its own
# flag column and, in the SVM cases, its own system/partition columns; this
# just normalizes and concatenates them -- no numbers are recomputed here.
# ----------------------------------------------------------------------
print("\n=== Table 4: accuracy by confound status ===")
confound_sources = [
    ("self_disclosure", RESULTS_DIR / "leakage_accuracy_comparison.csv", "self_referential_in_text", "LLM_fulltext_forced_json"),
    ("self_disclosure", RESULTS_DIR / "leakage_svm_accuracy_comparison.csv", "self_referential_in_text", None),
    ("topic_exclusivity", RESULTS_DIR / "icle_topic_accuracy_comparison.csv", "topic_exclusive_to_own_label", "LLM_fulltext_forced_json"),
    ("topic_exclusivity", RESULTS_DIR / "icle_topic_accuracy_comparison_svm.csv", "topic_exclusive_to_own_label", None),
]

confound_tables = []
for confound_name, path, flag_col, default_system in confound_sources:
    if not path.exists():
        print(f"Skipping {confound_name} ({path.name}): not found.")
        continue
    df = pd.read_csv(path).rename(columns={flag_col: "confound_present"})
    if "system" not in df.columns:
        df.insert(0, "system", default_system)
    if "partition" not in df.columns:
        df.insert(1, "partition", "all")
    df.insert(0, "confound", confound_name)
    confound_tables.append(df[["confound", "system", "partition", "confound_present", "n", "accuracy"]])

if confound_tables:
    confound_table = pd.concat(confound_tables, ignore_index=True)
    confound_table.to_csv(ASSETS_DIR / "table_confound_accuracy.csv", index=False)
    print(confound_table.to_string(index=False))
    print(f"Saved: {ASSETS_DIR / 'table_confound_accuracy.csv'}")
else:
    print("No confound-accuracy sources found. Run 11 and 12 first.")

# ----------------------------------------------------------------------
# Figures 3-8: confusion matrices, TOEFL11 test partition only (the
# corpus-design-controlled partition), masked vs. full-text, SVM and both
# LLM prompt formats (forced-JSON, the primary/headline condition, plus
# reasoning, to check whether forced-JSON's output patterns are a
# prompt-format artifact or a property of the underlying model). Computed
# directly from raw prediction files, not from any precomputed
# confusion-matrix CSV, so there's a single code path.
#
# NOTE: the SVM confusion matrices are built from the FULL TOEFL11 test set
# (n=100/language, from 04/07's full evaluation). The LLM confusion
# matrices can only be built from the 800-doc stratified eval subset
# (n=50/language for TOEFL11 within it, from 05/08/09) -- the LLM was never
# run on the full TOEFL11 test set. State this n difference explicitly in
# the paper's figure caption.
# ----------------------------------------------------------------------
print("\n=== Figures 3-8: TOEFL11 confusion matrices ===")


def plot_confusion(cm_df: pd.DataFrame, title: str, out_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(5.5, 5))
    im = ax.imshow(cm_df.values, cmap="Blues")
    ax.set_xticks(range(len(LANGS)))
    ax.set_xticklabels(LANGS, rotation=45, ha="right")
    ax.set_yticks(range(len(LANGS)))
    ax.set_yticklabels(LANGS)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title, fontsize=9)
    vmax = cm_df.values.max() if cm_df.values.size else 1
    for i in range(len(LANGS)):
        for j in range(len(LANGS)):
            val = cm_df.values[i, j]
            color = "white" if val > vmax * 0.6 else "black"
            ax.text(j, i, int(val), ha="center", va="center", fontsize=8, color=color)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(out_path, dpi=200)
    plt.close(fig)


svm_pred_sources = [
    ("SVM, masked, TOEFL11 test", RESULTS_DIR / "svm_predictions_full.csv", "fig_confusion_toefl_masked_svm.png"),
    ("SVM, full-text, TOEFL11 test", RESULTS_DIR / "svm_predictions_fulltext.csv", "fig_confusion_toefl_fulltext_svm.png"),
]
for title, path, out_name in svm_pred_sources:
    if not path.exists():
        print(f"Skipping confusion matrix ({title}): {path.name} not found.")
        continue
    df = pd.read_csv(path)
    df = df[df["partition"] == "toefl11_test"]
    if df.empty:
        print(f"Skipping confusion matrix ({title}): no toefl11_test rows in {path.name}.")
        continue
    cm = confusion_matrix(df["true_l1"], df["svm_predicted_l1"], labels=LANGS)
    cm_df = pd.DataFrame(cm, index=LANGS, columns=LANGS)
    out_path = ASSETS_DIR / out_name
    plot_confusion(cm_df, f"{title} (n={len(df)})", out_path)
    print(f"Saved: {out_path} (n={len(df)})")

llm_pred_sources = [
    ("LLM, masked, forced-JSON, TOEFL11 eval subset", RESULTS_DIR / "llm_predictions.csv", "fig_confusion_toefl_masked_llm.png"),
    ("LLM, full-text, forced-JSON, TOEFL11 eval subset", RESULTS_DIR / "llm_predictions_fulltext.csv", "fig_confusion_toefl_fulltext_llm.png"),
    # Reasoning-format counterparts of the two above. Added to check whether the forced-JSON
    # masked condition's output collapse (near-total DEU/SPA prediction, observed on a real run)
    # and the full-text condition's SPA over-prediction are properties of the underlying model,
    # or specific to the forced-JSON prompt format -- if reasoning collapses the same way, the
    # prior is model-level; if it spreads out more, forced-JSON is suppressing exploration.
    ("LLM, masked, reasoning, TOEFL11 eval subset", RESULTS_DIR / "llm_predictions_masked_reasoning.csv", "fig_confusion_toefl_masked_llm_reasoning.png"),
    ("LLM, full-text, reasoning, TOEFL11 eval subset", RESULTS_DIR / "llm_predictions_fulltext_reasoning.csv", "fig_confusion_toefl_fulltext_llm_reasoning.png"),
]
for title, path, out_name in llm_pred_sources:
    if not path.exists():
        print(f"Skipping confusion matrix ({title}): {path.name} not found.")
        continue
    df = pd.read_csv(path)
    df = df[
        (df["eval_partition"] == "toefl11_test")
        & df["error"].fillna("").eq("")
        & df["predicted_l1"].notna()
    ]
    if df.empty:
        print(f"Skipping confusion matrix ({title}): no usable toefl11_test rows in {path.name}.")
        continue
    cm = confusion_matrix(df["true_l1"], df["predicted_l1"], labels=LANGS)
    cm_df = pd.DataFrame(cm, index=LANGS, columns=LANGS)
    out_path = ASSETS_DIR / out_name
    plot_confusion(cm_df, f"{title} (n={len(df)})", out_path)
    print(f"Saved: {out_path} (n={len(df)})")

print(f"\nAll paper assets saved to: {ASSETS_DIR}")
